# Stage1 分钟事件检查点机械一致性

假设：在已冻结的逐年 f696+b472 自身账户上，09:47 后保存完整状态，HOLD 与单次分钟买入的后续 20 个交易日应和从起点回放逐项一致。本卡只验证机制，不产生经济标签或超额结论。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_episode import replay_event_full_path
from src.backtest.stage1_event_checkpoint import capture_after_open,resume_episode
from src.backtest.ibkr_tiered import settle_tiered_event
from src.backtest.event_guard_clock import guard_slot
from copy import deepcopy
import exchange_calendars as xc
from qlib.data import D
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
root=Path.cwd();card=root/"research/stage1_event_checkpoint_parity_20261029"
raw=(card/"plan.json").read_bytes();plan=json.loads(raw);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["full_replay_code"])==plan["full_replay_code_sha256"]
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"] and sha(old/"plan.json")==plan["old_source_plan_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
minute_plan=json.loads((root/plan["minute_source_plan"]).read_text())
assert sha(root/minute_plan["provider_uri"]/"qlib_store_manifest.json")==minute_plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/minute_plan["provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{year}" for year in parent_plan["signal_years"]],*[f"models/recovery_{year}" for year in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and len(book["authority"])==len(frame) and book["audit"]["status"]=="passed"
open_models={year:DatedModel(objects[f"models/open_{year}"],f"{year-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{year}") for year in parent_plan["signal_years"]}
recovery_models={}
for year in parent_plan["signal_years"]:
    if year in parent_plan["explicit_no_recovery_model_years"]:
        recovery_models[year]=DatedModel(None,f"{year-1}-12-31",f"explicit_no_model_{year}_insufficient_prior_teacher",explicit_no_model=True)
    else:
        recovery_models[year]=DatedModel(objects[f"models/recovery_{year}"],f"{year-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{year}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])


In [2]:

# The minute is chosen by source and actual whole-share fundability only.
day=pd.Timestamp(plan["event_day"])
checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,
    recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan)
assert checkpoint.state.q.shape==(2,)
bars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],
    start_time=str(day.date())+" 09:31:00",end_time=str(day.date())+" 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
    item=bars.xs(symbol,level="instrument").copy()
    item.columns=[symbol+"_"+field.removeprefix("$") for field in item]
    parts.append(item)
quotes=pd.concat(parts,axis=1).sort_index()
assert quotes[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all()
schedule=xc.get_calendar("XNYS",start=day-pd.Timedelta(days=1),end=day+pd.Timedelta(days=1)).schedule
close=pd.Timestamp(schedule.loc[day]["close"]).tz_convert("America/New_York").tz_localize(None)
selected=None
for minute in quotes.itertuples():
    slot=guard_slot(minute.Index,session_open=day+pd.Timedelta(hours=9,minutes=30),
        session_close=close,original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
    if slot is None:continue
    mark=np.array([minute.SOXL_close,minute.SOXX_close],float)
    nav=float(checkpoint.state.q@mark+checkpoint.state.cash)
    weights=np.r_[checkpoint.state.q*mark,checkpoint.state.cash]/nav
    room=min(max(0.,weights[2]-.15),3*max(0.,.30-weights[0]),max(0.,1-weights[1]))
    d=min(room,plan["maximum_beta_increment"])
    if d<=1e-10:continue
    targets={"SOXX":weights+np.array([0.,d,-d]),"SOXL":weights+np.array([d/3,0.,-d/3])}
    assert abs((3*targets["SOXX"][0]+targets["SOXX"][1])-(3*targets["SOXL"][0]+targets["SOXL"][1]))<1e-8
    fill_quote=quotes.loc[slot.fill_at+pd.Timedelta(minutes=1)]
    fill=np.array([fill_quote.SOXL_open,fill_quote.SOXX_open],float)
    raw_fill=np.array([fill_quote.SOXL_raw_open,fill_quote.SOXX_raw_open],float)
    funded={}
    for symbol,target in targets.items():
        trade=settle_tiered_event(checkpoint.state.q,checkpoint.state.cash,desired=target,
            adjusted_prices=fill,raw_prices=raw_fill,stamp=slot.fill_at,fees=deepcopy(checkpoint.state.fees))
        after_nav=float(trade["quantities"]@fill+trade["cash"])
        legal=bool(after_nav>0 and trade["cash"]/after_nav>=.15-1e-8 and
            trade["quantities"][0]*fill[0]/after_nav<=.30+1e-8)
        funded[symbol]=legal and len(trade["orders"])==1 and trade["orders"][0]["symbol"]==symbol and not trade["orders"][0]["sell"]
    if funded["SOXL"] and funded["SOXX"]:
        selected={"trade_date":day,"bar_end_at":slot.bar_end_at,"decision_at":slot.decision_at,
            "fill_at":slot.fill_at,"desired_weights":targets["SOXX"],
            "adjusted_fill_prices":fill,"raw_fill_prices":raw_fill}
        break
assert selected is not None
source_event={k:(str(v) if isinstance(v,pd.Timestamp) else np.asarray(v).tolist()) for k,v in selected.items()}
(card/"selected_source_event.json").write_text(json.dumps(source_event,indent=2)+"\n")
print("source_event",json.dumps(source_event))


In [3]:

# HOLD must equal the frozen parent; EVENT continuation must equal the independent full replay.
from pandas.testing import assert_frame_equal
hold=resume_episode(frame,checkpoint,controller_by_date=controller,recovery_spec=recovery_spec,
    action_spec=action_spec,selection=prepared["selection"],plan=parent_plan,event=None,horizon_sessions=20)
start=checkpoint.index
parent_slice=book["independent_replay"].iloc[start:start+20].reset_index(drop=True)
assert_frame_equal(hold["replay"].reset_index(drop=True),parent_slice,
    check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
order_days=set(hold["replay"].trade_date)
parent_orders=book["orders"].loc[book["orders"].trade_date.isin(order_days)].reset_index(drop=True)
assert_frame_equal(hold["orders"].reset_index(drop=True),parent_orders,
    check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
assert_frame_equal(hold["targets"].reset_index(drop=True),book["targets"].iloc[start:start+20].reset_index(drop=True),
    check_dtype=False,rtol=0,atol=plan["maximum_weight_error"])
event_episode=resume_episode(frame,checkpoint,controller_by_date=controller,recovery_spec=recovery_spec,
    action_spec=action_spec,selection=prepared["selection"],plan=parent_plan,event=selected,horizon_sessions=20)
full=replay_event_full_path(frame,models=None,plan=parent_plan,slippage_bps=0,
    controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,
    selection=prepared["selection"],event=selected)
full_slice=full["replay"].iloc[start:start+20].reset_index(drop=True)
assert_frame_equal(event_episode["replay"].reset_index(drop=True),full_slice,
    check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
full_orders=full["orders"].loc[full["orders"].trade_date.isin(order_days)].reset_index(drop=True)
assert_frame_equal(event_episode["orders"].reset_index(drop=True),full_orders,
    check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
assert_frame_equal(event_episode["targets"].reset_index(drop=True),full["targets"].iloc[start:start+20].reset_index(drop=True),
    check_dtype=False,rtol=0,atol=plan["maximum_weight_error"])
assert len(event_episode["orders"].query('slot == "event"'))==1
full_prefix=replay_event_full_path(frame.iloc[:start+20],models=None,plan=parent_plan,slippage_bps=0,
    controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,
    selection=prepared["selection"],event=selected)
assert event_episode["monthly_fee_shares"]==full_prefix["monthly_fee_shares"]
assert_frame_equal(event_episode["replay"].reset_index(drop=True),full_prefix["replay"].iloc[start:].reset_index(drop=True),
    check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
errors={c:float((event_episode["replay"][c]-full_slice[c]).abs().max()) for c in
    ["native_expected_nav","ending_cash","ending_soxl_quantity","ending_soxx_quantity","fees","strategy_return"]}
audit={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw).hexdigest(),
    "decision":"checkpoint_20_session_mechanical_parity_pass",
    "parent_recorder_id":plan["parent_recorder_id"],"source_readback_entries":len(receipt),
    "event_day":str(day.date()),"bar_end_at":str(selected["bar_end_at"]),
    "horizon_sessions":20,"hold_orders":len(hold["orders"]),"event_orders":len(event_episode["orders"]),
    "max_event_errors":errors,"new_model_fits":0,"new_candidate_accounts":0,
    "historical_research_passed":False,"dividend_receivable_parity":"unknown"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(audit,ensure_ascii=False,indent=2))


## 结论

2019-12-19首个双动作可整股成交的09:48分钟，09:49可决策、09:50成交。HOLD续跑与封存父本、单次SOXX事件续跑与从起点完整回放，在后续20会话的逐日订单、目标、现金、持仓、Tiered费、风险状态、NAV均一致；最大差0，月度计数亦一致。来源为Qlib分钟库、封存逐年模型和原生账户Recorder。此卡只通过机械门，无收益标签、因子拟合或SOXX超额。调整价下单独应收分红仍为unknown。下一步冻结三分支费后标签及逐年信息门。